In [3]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL = 'gpt-6-luna'

#### Images and vision
- vision : 모델이 이미지를 입력으로 받아 이해하는 능력
- response api 를 이용
- image api 이용

In [14]:
# 이미지 생성

from openai import OpenAI
import base64

client = OpenAI()

response = client.responses.create(
    model=MODEL,
    input="Generate an image of gray cat hugging an otter with an orange scarf",
    tools=[{"type":"image_generation"}]
)

# 생성된 이미지 저장
image_data = [output.result for output in response.output if output.type == "image_generation_call"]

if image_data:
    image_base64 = image_data[0]
    with open("./data/cat_and_otter.png", "wb")as f:
        f.write(base64.b64decode(image_base64))


In [5]:
# 웹 이미지 분석

content = [
    {"type":"input_text", "text":"what's in this image?"},
    {"type":"input_image", "image_url":"https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQ7jJtnf3o9TquaQmFllHDz1bsOivgtaNNsUfTH3CYu7A&s=10"},
]

response = client.responses.create(
    model=MODEL,
    input=[
        {"role":"user", "content":content}
    ],
    tools=[{"type":"image_generation"}]
)

print(response.output_text)

A crowded sandy beach on a sunny day, with colorful umbrellas and many people swimming in clear blue water. A tree-covered headland and distant hills line the coast, and a small boat is visible offshore.


In [9]:
# 로컬 이미지 파일 분석

# rb에서 b 는 text 파일이 아닌 경우에 붙임
def encode_image(image_path):
    with open(image_path, "rb")as f:
        return base64.b64encode(f.read()).decode("utf-8")


In [11]:
image_path = "./data/bottle.png"

# 이미지 로드
base64_image = encode_image(image_path)

content = [
    {"type":"input_text", "text":"what's in this image?"},
    {"type":"input_image", "image_url":f"data:image/png;base64,{base64_image}"},
]

response = client.responses.create(
    model=MODEL,
    input=[
        {"role":"user", "content":content}
    ],
)

print(response.output_text)

A yellow insulated water bottle with a silver cap, shown against a plain white background.


In [ ]:
# file id 이용
# purpose="vision"  => 파인튜닝 용도
# purpose="user_data"  => 일반적인 철 상황

def create_file(file_path):
    with open(file_path, "rb") as f:
        result = client.files.create(file=f, purpose="vision")
        return result.id



In [17]:
image_path = "./data/bottle.png"

# 이미지 로드
file_id = create_file(image_path)

content = [
    {"type":"input_text", "text":"what's in this image?"},
    {"type":"input_image", "file_id":file_id},
]

response = client.responses.create(
    model=MODEL,
    input=[
        {"role":"user", "content":content}
    ],
)

print(response.output_text)

A pale yellow reusable bottle with a shiny silver cap, shown against a plain white background.


#### 이미지 전용 모델

In [20]:
response = client.images.generate(
    model="gpt-image-2",
    prompt="A cute red panda reading a book, watercolor style",
    size="auto",      # 1024 x 1024, 1024 x 1536, 1536 x 1024
    quality="high",      # low, medium, high, auto
    output_format="png"
)

image_base64 = response.data[0].b64_json
image_byte = base64.b64decode(image_base64)

with open("./data/panda.png", "wb") as f:
    f.write(image_byte)



#### Audio and Speech
- 음성 비서
- 실시간 음성 처리
- TTS (Text-to-Speech)
- STT (Speech-to-Text)

In [30]:
# TTS

speech = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="fable",
    input="AI 인공지능 TTS 입니다람쥐. 나 배고파",
    speed=1.5,
    response_format="mp3"
)

with open("./data/tts1.mp3", "wb")as f:
    f.write(speech.read())

In [24]:
# 음성 => 텍스트 변환

with open("./data/speech.mp3", "rb")as f:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=f
    )

print(transcription.text)

안녕하세요. 이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다. GPT API에 대해서 생소하신 분들도 있을 텐데, 우리가 잘 알고 있는 챗GPT, 챗GPT의 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러 가지들이 있는데, 좀 이 강의의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챗봇 프로그램만 만드는 게 사실 쉽지는 않은데, 이거를 어떻게 해서 구현을 하는지, 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로, 예제는 여러 가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고.


In [25]:
with open("./data/rag.mp3", "rb")as f:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        language="en",
        prompt="이 mp3 파일은 AI 개발 관련 강의야.",
        file=f
    )

print(transcription.text)

Large language models, they are everywhere. They get some things amazingly right and other things very interestingly wrong. My name is Marina Danilevsky. I am a senior research scientist here at IBM Research, and I want to tell you about a framework to help large language models be more accurate and more up to date: retrieval augmented generation, or RAG. Let's just talk about the generation part for a minute. So forget the retrieval augmented. So the generation, this refers to large language models, or LLMs, that generate text in response to a user query, referred to as a prompt. These models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: In our solar system, what planet has the most moons? And my response was, Oh, that's really great that you're asking me this question. I loved space when I was your age. Of course, that was like 30 years ago, but I know this. I read an article, and the article s

In [29]:
# 음성을 문장별로 분리 / 시간 정보

with open("./data/rag.mp3", "rb")as f:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        language="en",
        prompt="이 mp3 파일은 AI 개발 관련 강의야.",
        file=f,
        response_format="verbose_json",
        timestamp_granularities=['segment']
    )

for segment in transcription.segments:
    print(f"{segment.start:.2f} ~ {segment.end:.2f}", segment.text)

0.00 ~ 2.64  Large language models, they are everywhere.
2.64 ~ 5.04  They get some things amazingly right
5.04 ~ 7.90  and other things very interestingly wrong.
7.90 ~ 9.36  My name is Marina Danilevsky.
9.36 ~ 12.14  I am a senior research scientist here at IBM Research.
12.14 ~ 14.50  And I want to tell you about a framework
14.50 ~ 16.40  to help large language models
16.40 ~ 18.60  be more accurate and more up-to-date.
18.60 ~ 22.60  Retrieval Augmented Generation, or RAG.
22.60 ~ 24.54  Let's just talk about the generation part for a minute.
24.54 ~ 26.60  So forget the retrieval augmented.
26.84 ~ 30.20  So the generation, this refers to large language models,
30.20 ~ 34.66  or LLMs, that generate text in response to a user query
34.66 ~ 36.16  referred to as a prompt.
36.16 ~ 38.30  These models can have some undesirable behavior.
38.30 ~ 41.20  I want to tell you an anecdote to illustrate this.
41.20 ~ 44.64  So my kids, they recently asked me this question.
44.64 ~ 49.14  In

In [13]:
# 1. 음성 파일을 텍스트로 변환 : chunk.mp3
# 2. 텍스트로 변환한 내용을 기반으로 답변 생성
# 3. 답변을 음성으로 변환 answer.mp3

from openai import OpenAI
import base64

client = OpenAI()

with open("./data/chunk.mp3", "rb") as f:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=f
    )

print(transcription.text)



context = [
    {"role":"system", "content":"내가 주는 텍스트를 가지고 질문에 대한 답변을 한국어로 해줘."},
    {"role":"user", "content":transcription.text},
    {"role":"user", "content":"그럼 청크는 어느 분야에서 사용하기 위한 개념이야?"}
]

response = client.responses.create(
    model=MODEL,
    input=context,
    stream=False
)

print(response.output_text)


speech = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="marin",
    input=response.output_text,
    speed=1.5,
    response_format="mp3"
)

with open("./data/tts2.mp3", "wb") as f:
    f.write(speech.read())


첫 번째 이해해야 되는 개념이 바로 청크입니다. 그래서 문서를 나누는 게 바로 청크이죠. 긴 문서를 검색 증강 생성, RAG에서 다루기 좋게 적당한 길이로 의미 단위로 자르는 것, 바로 이 청크가 되겠습니다. 일반적으로는 한 200에서 한 800 토큰 정도로 우리가 이 청크를 만들게 되고요. 그리고 단락이나 섹션이나 표 이런 문서 구조를 유지한 상태로 나누는 것이 이상적이다라고 얘기를 하죠. 그리고 문서 내에 단락이나 섹션들이 있을 겁니다. 그래서 의미가 사실은 유지가 되어야 되기 때문에, 그리고 단락별로, 문장별로, 크게는 섹션별로 청크를 하게 되면은 의미가 좀 더 정확하게 유지가 되겠죠.
청크는 긴 문서를 적당한 길이의 의미 단위로 나눠, **검색 증강 생성(RAG)**에서 검색하고 활용하기 쉽게 만들기 위한 개념입니다. 문서의 단락이나 섹션 같은 구조를 유지하면 의미를 더 정확하게 보존할 수 있습니다.
